# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srilaya30/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [38]:
from pathlib import Path
import pandas as pd
import numpy as np

# Repository
repo_root = Path("/content/flyrank-ml-internship")

# Dataset
DATA_PATH = (
    repo_root
    / "data"
    / "raw"
    / "content_refresh_anonymized.csv"
)

# Output
OUTPUT_PATH = (
    repo_root
    / "work"
    / "outputs"
    / "baseline_action_score.csv"
)

# Checks
print("Repository exists:", repo_root.exists())
print("Dataset exists:", DATA_PATH.exists())
print("Dataset path:", DATA_PATH)

if not repo_root.exists():
    raise FileNotFoundError(
        "Repository not found at /content/flyrank-ml-internship"
    )

if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {DATA_PATH}"
    )

# Create output directory
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

# Load dataset
df = pd.read_csv(DATA_PATH)

print("\nDataset loaded successfully!")
print("Dataset shape:", df.shape)

Repository exists: True
Dataset exists: True
Dataset path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

Dataset loaded successfully!
Dataset shape: (30000, 44)


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


### Baseline rule

I prioritize webpages for content-refresh review using two observable signals:

1. **Staleness:** webpages that have gone longer without an update receive a higher priority.
2. **CTR opportunity:** webpages with relatively weak CTR receive a higher priority.

The baseline is a decision-support ranking, not a prediction of future performance.

The score combines percentile ranks of the two signals:

- 60% weight: staleness
- 40% weight: CTR opportunity

A higher score means the webpage should be reviewed earlier.

### Reason codes

The rule produces one reason code for each webpage:

- `STALE_CONTENT` — staleness is the main reason for the review.
- `CTR_OPPORTUNITY` — weaker CTR is the main reason for the review.

### Action

The action for every ranked webpage is:

`REVIEW_REFRESH`

This is a baseline rule only. It does not use product flags, future-window information, or the trend label as an input.

In [39]:
# ============================================================
# Signal checks
# ============================================================

# Convert required columns to numeric
df["days_since_last_update"] = pd.to_numeric(
    df["days_since_last_update"],
    errors="coerce"
)

df["ctr"] = pd.to_numeric(
    df["ctr"],
    errors="coerce"
)

df["impressions_90d"] = pd.to_numeric(
    df["impressions_90d"],
    errors="coerce"
)

# ============================================================
# SIGNAL 1: STALENESS
# ============================================================

df["staleness_bucket"] = pd.cut(
    df["days_since_last_update"],
    bins=[-1, 30, 90, 180, 365, np.inf],
    labels=[
        "0-30",
        "31-90",
        "91-180",
        "181-365",
        "365+"
    ]
)

staleness_table = (
    df.groupby(
        "staleness_bucket",
        observed=False
    )
    .agg(
        n=("content_id", "size"),
        avg_impressions=("impressions_90d", "mean"),
        avg_ctr=("ctr", "mean")
    )
    .reset_index()
)

print("SIGNAL CHECK 1 — STALENESS")
display(staleness_table)

# ============================================================
# SIGNAL 2: CTR
# ============================================================

df["ctr_bucket"] = pd.qcut(
    df["ctr"],
    q=4,
    duplicates="drop"
)

ctr_table = (
    df.groupby(
        "ctr_bucket",
        observed=False
    )
    .agg(
        n=("content_id", "size"),
        avg_impressions=("impressions_90d", "mean"),
        avg_staleness=("days_since_last_update", "mean")
    )
    .reset_index()
)

print("\nSIGNAL CHECK 2 — CTR")
display(ctr_table)

SIGNAL CHECK 1 — STALENESS


,staleness_bucket,n,avg_impressions,avg_ctr
0,0-30,20480,4199.614062,0.609021
1,31-90,175,6506.748571,0.117543
2,91-180,9171,7486.665140,0.238367
3,181-365,169,1206.893491,3.210828
4,365+,5,8.200000,20.000000



SIGNAL CHECK 2 — CTR


,ctr_bucket,n,avg_impressions,avg_staleness
0,"(-0.001, 0.07]",15224,1910.121322,43.051104
1,"(0.07, 0.29]",7503,9335.023857,51.741970
2,"(0.29, 100.0]",7273,7822.166644,46.654613


### Signal verdicts

**Staleness — MIXED**

Observed result: The staleness buckets do not show a consistent relationship with average CTR. The 31–90 and 91–180 day groups have lower average CTR than the 0–30 day group, while the 181–365 and 365+ groups have much higher average CTR. The oldest groups also have very small sample sizes (`n=169` and `n=5`).

Reason: The measured relationship is mixed, so staleness is retained as a directional decision-support signal rather than treated as proof that older pages always have weaker CTR.

**CTR — MIXED**

Observed result: The CTR buckets do not show a monotonic relationship with average staleness. Average staleness is 43.05, 51.74, and 46.65 days across the three observed CTR buckets.

Reason: The relationship is mixed, so CTR is retained as a directional opportunity signal rather than treated as a strong standalone indicator.

The staleness signal is linked to the refresh/staleness flag logic discussed in the session. Both checks are treated as measured signal audits, not as target labels.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*


### Queue construction

For each webpage, I calculate two percentile-based signals:

- `staleness_score`: higher when the webpage has been longer since its last update.
- `ctr_opportunity_score`: higher when CTR is relatively low.

The final baseline score is:

`baseline_score = 0.60 × staleness_score + 0.40 × ctr_opportunity_score`

The queue is ranked from highest to lowest baseline score.

This is intentionally simple and interpretable so that a future ML model can be compared against this baseline.

In [40]:
# ============================================================
# Build ranked queue
# ============================================================

score_df = df.copy()

# Keep rows that can be scored
score_df = score_df.dropna(
    subset=[
        "content_id",
        "client_id",
        "days_since_last_update",
        "ctr"
    ]
).copy()

# ============================================================
# Staleness score
# ============================================================

score_df["staleness_score"] = (
    score_df["days_since_last_update"]
    .rank(
        method="average",
        pct=True
    )
)

# ============================================================
# CTR opportunity score
# Lower CTR = higher opportunity
# ============================================================

score_df["ctr_opportunity_score"] = (
    1
    - score_df["ctr"].rank(
        method="average",
        pct=True
    )
)

# ============================================================
# Baseline score
# ============================================================

score_df["baseline_score"] = (
    0.60 * score_df["staleness_score"]
    + 0.40 * score_df["ctr_opportunity_score"]
)

# ============================================================
# One reason code
# ============================================================

score_df["reason_code"] = np.where(
    score_df["staleness_score"]
    >= score_df["ctr_opportunity_score"],
    "STALE_CONTENT",
    "CTR_OPPORTUNITY"
)

# ============================================================
# Action
# ============================================================

score_df["action"] = "REVIEW_REFRESH"

# ============================================================
# Rank
# ============================================================

queue = (
    score_df[
        [
            "content_id",
            "client_id",
            "baseline_score",
            "reason_code",
            "action"
        ]
    ]
    .sort_values(
        ["baseline_score", "content_id"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

queue["rank"] = np.arange(
    1,
    len(queue) + 1
)

queue = queue[
    [
        "rank",
        "content_id",
        "client_id",
        "baseline_score",
        "reason_code",
        "action"
    ]
]

print("Ranked queue shape:", queue.shape)

display(queue.head(20))

Ranked queue shape: (30000, 6)


,rank,content_id,client_id,baseline_score,reason_code,action
0,1,content_55a5b1c46474,client_4ec9599fc2,0.911893,STALE_CONTENT,REVIEW_REFRESH
1,2,content_f6fdf87348f6,client_4ec9599fc2,0.911893,STALE_CONTENT,REVIEW_REFRESH
2,3,content_1b4ec72dafd4,client_4ec9599fc2,0.911843,STALE_CONTENT,REVIEW_REFRESH
3,4,content_8d56efff1e71,client_4ec9599fc2,0.911843,STALE_CONTENT,REVIEW_REFRESH
4,5,content_06e19c6486b0,client_4ec9599fc2,0.911783,STALE_CONTENT,REVIEW_REFRESH
5,6,content_e2b702f4f92b,client_4ec9599fc2,0.911783,STALE_CONTENT,REVIEW_REFRESH
6,7,content_02b0d6e30129,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH
7,8,content_6476d1d8c050,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH
8,9,content_7a888d3d99c8,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH
9,10,content_94991fe6268c,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH


In [41]:
# ============================================================
# Write baseline CSV
# ============================================================

OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

queue.to_csv(
    OUTPUT_PATH,
    index=False
)

print("CSV successfully written!")
print("Output path:", OUTPUT_PATH)
print("Rows written:", len(queue))

CSV successfully written!
Output path: /content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Rows written: 30000


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*



For each of the top 20 webpages, I record the action, reason code, confidence note, and what would make the recommendation wrong.

These are manual decision-support observations, not ground-truth labels.

In [42]:
# ============================================================
# Top-20 review data
# ============================================================

top20 = (
    queue.head(20)
    .merge(
        score_df[
            [
                "content_id",
                "client_id",
                "days_since_last_update",
                "ctr",
                "impressions_90d",
                "staleness_score",
                "ctr_opportunity_score"
            ]
        ],
        on=[
            "content_id",
            "client_id"
        ],
        how="left"
    )
)

display(top20)

,rank,content_id,client_id,baseline_score,reason_code,action,days_since_last_update,ctr,impressions_90d,staleness_score,ctr_opportunity_score
0,1,content_55a5b1c46474,client_4ec9599fc2,0.911893,STALE_CONTENT,REVIEW_REFRESH,373,0.0,35,0.999967,0.779783
1,2,content_f6fdf87348f6,client_4ec9599fc2,0.911893,STALE_CONTENT,REVIEW_REFRESH,373,0.0,2,0.999967,0.779783
2,3,content_1b4ec72dafd4,client_4ec9599fc2,0.911843,STALE_CONTENT,REVIEW_REFRESH,372,0.0,2,0.999883,0.779783
3,4,content_8d56efff1e71,client_4ec9599fc2,0.911843,STALE_CONTENT,REVIEW_REFRESH,372,0.0,1,0.999883,0.779783
4,5,content_06e19c6486b0,client_4ec9599fc2,0.911783,STALE_CONTENT,REVIEW_REFRESH,334,0.0,10,0.999783,0.779783
5,6,content_e2b702f4f92b,client_4ec9599fc2,0.911783,STALE_CONTENT,REVIEW_REFRESH,334,0.0,30,0.999783,0.779783
6,7,content_02b0d6e30129,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH,313,0.0,176,0.999683,0.779783
7,8,content_6476d1d8c050,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH,313,0.0,304,0.999683,0.779783
8,9,content_7a888d3d99c8,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH,313,0.0,95,0.999683,0.779783
9,10,content_94991fe6268c,client_19581e27de,0.911723,STALE_CONTENT,REVIEW_REFRESH,313,0.0,7,0.999683,0.779783


### Top-20 manual review

1. **Rank 1** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 373 days since last update and a very high staleness score. Wrong if the page was intentionally left unchanged or the update signal does not reflect recent work.

2. **Rank 2** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 373 days since last update and a very high staleness score. Wrong if the content remains intentionally stable or the update date is incomplete.

3. **Rank 3** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 372 days since last update and a very high staleness score. Wrong if no meaningful refresh is needed.

4. **Rank 4** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 372 days since last update and a very high staleness score. Wrong if the content is still appropriate despite its age.

5. **Rank 5** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 334 days since last update and a very high staleness score. Wrong if the page has already been reviewed or intentionally kept unchanged.

6. **Rank 6** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 334 days since last update and a very high staleness score. Wrong if age does not indicate a real refresh opportunity.

7. **Rank 7** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 313 days since last update and a very high staleness score. Wrong if the content is still accurate and useful.

8. **Rank 8** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 313 days since last update and a very high staleness score. Wrong if there are no meaningful content changes available.

9. **Rank 9** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 313 days since last update and a very high staleness score. Wrong if the freshness signal does not represent the actual content state.

10. **Rank 10** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 313 days since last update and a very high staleness score. Wrong if the page remains appropriate without a refresh.

11. **Rank 11** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 305 days since last update and a very high staleness score. Wrong if the content is intentionally stable.

12. **Rank 12** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 305 days since last update and a very high staleness score. Wrong if the page does not have a meaningful refresh opportunity.

13. **Rank 13** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 305 days since last update and a very high staleness score. Wrong if the update timestamp does not reflect recent content work.

14. **Rank 14** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 305 days since last update and a very high staleness score. Wrong if the page remains useful and accurate.

15. **Rank 15** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 305 days since last update and a very high staleness score. Wrong if age alone does not indicate a refresh need.

16. **Rank 16** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 305 days since last update and a very high staleness score. Wrong if the page has already been reviewed outside the recorded update field.

17. **Rank 17** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 305 days since last update and a very high staleness score. Wrong if the page is intentionally stable.

18. **Rank 18** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 305 days since last update and a very high staleness score. Wrong if the page does not need substantive changes.

19. **Rank 19** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 305 days since last update and a very high staleness score. Wrong if the recorded update date is not representative of the current content.

20. **Rank 20** — Action: `REVIEW_REFRESH` — Reason: `STALE_CONTENT` — Confidence: Moderate because the page has 305 days since last update and a very high staleness score. Wrong if the page remains accurate and useful despite its age.

### Overall observation

All top 20 webpages receive the `REVIEW_REFRESH` action and `STALE_CONTENT` reason code. Their staleness scores are higher than their CTR opportunity scores, so staleness determines the reason code.

The CTR value is 0.0 for all top 20 rows, which also contributes to the CTR opportunity score.

These results should be interpreted as a review queue rather than evidence that every page definitely requires a refresh.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*


### Weak picks

I inspected the lowest-ranked pages as potential weak picks.

The baseline is intentionally simple, so a low-ranked page does not necessarily mean that it has no refresh opportunity.

The bottom 20 contains both `STALE_CONTENT` and `CTR_OPPORTUNITY` recommendations. The lowest-ranked row has a baseline score of approximately 0.004.

Potential weak picks can occur when:

- search intent changes are not represented;
- seasonality is not represented;
- the update timestamp does not fully represent content freshness;
- CTR is affected by factors not represented by the baseline;
- a page has a useful refresh opportunity that is not captured by these two signals.

These limitations mean the score should be used as directional decision-support rather than as a prediction of future performance.

### Leakage check

The baseline uses only:

- `days_since_last_update`
- `ctr`

I did not use product flags, `trend_direction`, target labels, or future-window information in the scoring formula.

In [43]:
# ============================================================
# Bottom 20 / weak picks
# ============================================================

weak_picks = queue.tail(20).copy()

print("Bottom 20 / weakest-ranked pages:")
display(weak_picks)

Bottom 20 / weakest-ranked pages:


,rank,content_id,client_id,baseline_score,reason_code,action
29980,29981,content_b36ab9f13f2b,client_9f14025af0,0.035387,STALE_CONTENT,REVIEW_REFRESH
29981,29982,content_a1c65f070bad,client_9f14025af0,0.035020,STALE_CONTENT,REVIEW_REFRESH
29982,29983,content_dfce82404813,client_9f14025af0,0.035020,STALE_CONTENT,REVIEW_REFRESH
29983,29984,content_4e95a8389562,client_9f14025af0,0.034440,STALE_CONTENT,REVIEW_REFRESH
29984,29985,content_b96873ca64c1,client_9f14025af0,0.034440,STALE_CONTENT,REVIEW_REFRESH
29985,29986,content_f26233911f33,client_9f14025af0,0.034440,STALE_CONTENT,REVIEW_REFRESH
29986,29987,content_9a7fe374c900,client_9f14025af0,0.034220,STALE_CONTENT,REVIEW_REFRESH
29987,29988,content_006b16e7a2e7,client_9f14025af0,0.034127,STALE_CONTENT,REVIEW_REFRESH
29988,29989,content_4272d3a330a3,client_9f14025af0,0.034127,STALE_CONTENT,REVIEW_REFRESH
29989,29990,content_cfa4d9f1bf0a,client_d4735e3a26,0.034127,STALE_CONTENT,REVIEW_REFRESH


In [45]:
# ============================================================
# Leakage check
# ============================================================

baseline_inputs = [
    "days_since_last_update",
    "ctr"
]

forbidden_inputs = [
    "trend_direction",
    "product_flag",
    "flag",
    "label",
    "target"
]

print("Signals used by baseline:")

for column in baseline_inputs:
    print(" -", column)

leaked_inputs = set(baseline_inputs).intersection(
    forbidden_inputs
)

print("\nForbidden inputs used:", leaked_inputs)

assert len(leaked_inputs) == 0

print("\n✓ No product flags or label-derived inputs used.")
print("✓ No future-window variable used in the scoring formula.")

Signals used by baseline:
 - days_since_last_update
 - ctr

Forbidden inputs used: set()

✓ No product flags or label-derived inputs used.
✓ No future-window variable used in the scoring formula.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [44]:
# ============================================================
# Final automated self-check
# ============================================================

print("========== ML-07 SELF-CHECK ==========")

print("Queue rows:", len(queue))

print(
    "Unique content IDs:",
    queue["content_id"].nunique()
)

print(
    "Missing scores:",
    queue["baseline_score"].isna().sum()
)

print(
    "Missing reason codes:",
    queue["reason_code"].isna().sum()
)

print(
    "Missing actions:",
    queue["action"].isna().sum()
)

print(
    "Scores sorted descending:",
    queue["baseline_score"].is_monotonic_decreasing
)

print(
    "CSV exists:",
    OUTPUT_PATH.exists()
)

assert len(queue) == 30000
assert queue["content_id"].nunique() == 30000
assert queue["baseline_score"].notna().all()
assert queue["reason_code"].notna().all()
assert queue["action"].notna().all()
assert queue["baseline_score"].is_monotonic_decreasing
assert OUTPUT_PATH.exists()

print("\n✓ ML-07 baseline checks passed.")

========== ML-07 SELF-CHECK ==========
Queue rows: 30000
Unique content IDs: 30000
Missing scores: 0
Missing reason codes: 0
Missing actions: 0
Scores sorted descending: True
CSV exists: True

✓ ML-07 baseline checks passed.
